# CookMatch — Colab Full Run

## Run order

1. **Runtime → Restart session** (if you re-ran cell 1 before and hit errors)
2. Run **cell 1** (downloads + syncs latest code from GitHub)
3. Run **cells 2 → 6** in this tab

Cell 1 always pulls fresh Python files from GitHub `main`, even if git clone is cached.

Flow: setup → Kaggle auth → load data → train → recommend → ablation.

## Design Rationale — Why We Built It This Way

### Problem
Standard collaborative filtering (CF) answers one question: *what did similar users like?*  
For food, that is not enough. A vegan user must never see meat recipes. A user with 20 minutes cannot cook a 3-hour dish. CF ignores both constraints entirely.

### Why a 3-stage cascade and not a single model?
1. **Safety must be a hard guarantee.** A unified neural model learns a soft trade-off between safety and relevance — it could still recommend a meat recipe to a vegan user if that recipe scores highly on other signals. The cascade makes violations structurally impossible: unsafe recipes are removed before ranking.
2. **Query context is session-level, not trainable offline.** Pantry contents, available time, and meal goal change every session. A static trained model cannot use them. Stage 3 applies them at query time with no retraining.
3. **Modularity enables rigorous ablation.** Each stage can be validated independently. If combined-mode fails, we know exactly which component to debug.

### Why Food.com?
Most RS datasets (MovieLens, Amazon) are ratings-only. Food.com is one of very few large public datasets that combines:
- User–recipe ratings (for CF)
- Structured ingredient lists (for Stage 1 safety filter + Stage 3 pantry signal)
- Cook time in minutes (for Stage 3 time signal)
- Meal tags (for Stage 3 intent signal)

All four fields are required. Food.com was the only viable option at this scale.

### Why keyword matching for safety (not ML)?
An ML allergen classifier that is 99% accurate still allows 1 in 100 violations. For allergens, that is unacceptable. Keyword matching gives 100% recall on known terms — fast, transparent, and auditable. The trade-off: coverage gaps for brand names and rare ingredients (documented as a limitation).


In [13]:
# 1) Bootstrap latest code from GitHub (run once per session)
import os

os.chdir("/content")

# Fetch bootstrap module from GitHub main FIRST (not from a stale clone)
import urllib.request

_RAW = "https://raw.githubusercontent.com/YUV3571/cookmatch-recipe-recommender/main/colab_init.py"
urllib.request.urlretrieve(_RAW, "/content/_cookmatch_colab_init.py")

import importlib.util

_spec = importlib.util.spec_from_file_location("colab_init", "/content/_cookmatch_colab_init.py")
ci = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(ci)

REPO_DIR = ci.bootstrap_repo()
ci.verify_eval_stack()

print("Code OK: latest eval fixes loaded")
print("GitHub ↔ Colab OK | repo:", REPO_DIR)
print("Next: run cells 2 → 6")
!ls {REPO_DIR}/src/data/

Connected via git clone


AssertionError: 

In [ ]:
# 2) Install dependencies
!pip install -q kagglehub pandas numpy scipy pyarrow

In [ ]:
# 3) Kaggle authentication
import os

# Option A (recommended): Kaggle API token
# Kaggle → Settings → API → Create New Token → copy KGAT_... value
os.environ["KAGGLE_API_TOKEN"] = "KGAT_your_token_here"  # paste your token

# Option B: upload kaggle.json instead (comment out Option A first)
# from google.colab import files
# uploaded = files.upload()
# !mkdir -p ~/.kaggle
# !mv kaggle.json ~/.kaggle/kaggle.json
# !chmod 600 ~/.kaggle/kaggle.json

assert os.environ.get("KAGGLE_API_TOKEN", "").startswith("KGAT_"), "Set a valid KAGGLE_API_TOKEN"
print("Kaggle token configured.")

In [ ]:
# 4) Load dataset
import os
import sys
import time

REPO_DIR = "/content/cookmatch-recipe-recommender"
import colab_init
colab_init.bind(REPO_DIR)

# --- RUN MODE ---
FULL_CATALOG = True      # False = quick 5000-row smoke test
FAST_ABLATION = True     # 30k eval catalog + 100 users (~5–15 min). False = hours on 231k
EVAL_CATALOG_SIZE = 30_000
USER_SAMPLE = 100 if FAST_ABLATION else 500
RECIPE_LIMIT = None if FULL_CATALOG else 5000
# ----------------

import kagglehub
from src.data.loader import (
    build_eval_recipe_catalog,
    get_dataset_path,
    load_interaction_split,
    load_recipes,
)

dataset_path = get_dataset_path()
print("Dataset path:", dataset_path)

t0 = time.time()
recipes = load_recipes(
    nrows=RECIPE_LIMIT,
    columns=["id", "name", "ingredients", "minutes", "tags"],
)
train = load_interaction_split("train")
validation = load_interaction_split("validation")

recipes_eval = build_eval_recipe_catalog(
    recipes, train, validation, max_recipes=EVAL_CATALOG_SIZE
)
recipes_train = recipes_eval if FAST_ABLATION else recipes

print(f"recipes loaded: {len(recipes)} ({'FULL' if FULL_CATALOG else 'SAMPLE'})")
print(f"train/eval catalog: {len(recipes_train)} recipes")
print(f"train interactions: {len(train)}")
print(f"validation rows: {len(validation)}")
print(f"ablation users: {USER_SAMPLE}")
print(f"load time: {time.time() - t0:.1f}s")

## Iteration Story — What We Tried and Why We Changed It

The system was built in three evaluated stages. Each stage was measured before deciding to add the next.

### Step 1 — Popularity baseline
Before training any ML model, we ran a popularity-only recommender (Bayesian-smoothed average rating). Hit@10 = 0.00 at catalog scale.  
**Decision:** Popularity alone is not viable. Established the baseline that every method must beat.

### Step 2 — MF-SVD standalone
We trained truncated SVD (20 factors) on catalog interactions and evaluated Hit@10 without content signals.  
Hit@10 = 0.00 even when restricted to MF's own top-5,000 candidates (tested at pool_k = 500, 1000, 2000, 5000 — all zero).  
**Finding:** CF alone is insufficient at this catalog scale. The eval catalog slice is too sparse for SVD to generalise (~5 ratings per recipe average).  
**Decision:** CF provides the personalisation scaffold but cannot do retrieval alone. Build Stage 3.

### Step 3 — Oracle ablation to validate Stage 3 signals
Before combining signals, each content signal was evaluated in isolation with oracle context (perfect pantry/time/intent built from the target recipe). Each hit 1.00 independently.  
**Decision:** All three signals work. Combine them.

### Issues found and fixed during iteration
| Issue | Fix |
|-------|-----|
| Oracle rows showing 0.00 | CF weight (45%) was drowning content signals → switched oracle eval to signal-only weights |
| Oracle pin_recipe appended at end of shortlist | Changed to prepend → oracle recovered to 1.00 |
| 375-min recipe in '30 min' query | Added hard time cap for time-mode panel |
| Kielbasa appearing for vegan user | Added explicit keyword to MEAT_KEYWORDS + regression test |
| Ablation taking 50+ min | Redesigned to 30k eval catalog + 500-candidate rerank pool → ~4 min |
| All-zero metrics on first ablation run | Discovered module cache in Colab serving stale code → added purge_cached_modules() |


In [ ]:
# 5) Train Stage 3 recommender + demo recommendations
import colab_init
colab_init.bind("/content/cookmatch-recipe-recommender")

from src.models.user_profile import UserProfile
from src.models.session_context import SessionContext
from src.recommend.stage3 import Stage3Recommender

recipes_train = globals().get("recipes_train", recipes)

from src.data.loader import filter_interactions_to_catalog

train_catalog = filter_interactions_to_catalog(train, recipes_train)
recommender = Stage3Recommender().fit(recipes_train, train_catalog)
print(f"trained on {len(recipes_train)} recipes, {len(train_catalog)} interactions")

profile = UserProfile(diet="vegan", allergens=["nuts", "dairy", "gluten"])
context = SessionContext(pantry=["tomato", "pasta", "garlic"], max_minutes=30, meal_intent="main")
known_user = int(train["user_id"].iloc[0])

recs = recommender.recommend(profile, context, user_id=known_user, top_n=5)
for rec in recs:
    print(f"{rec.final_score:.3f} | {rec.name}")
    print(f"  why: {rec.explanation}")

In [ ]:
# 6a) Demo — Panel A: Pantry mode
# Signal: what can I cook with what I have?
# Validated by stage3_oracle_pantry: Hit@10 = 1.0
import colab_init
colab_init.bind("/content/cookmatch-recipe-recommender")

from config.query_modes import QUERY_MODES
from src.models.user_profile import UserProfile
from src.models.session_context import SessionContext

assert 'recommender' in globals(), 'Run cell 5 first'

profile = UserProfile(diet='vegan', allergens=['nuts', 'dairy', 'gluten'])
context = SessionContext(pantry=['tomato', 'pasta', 'garlic', 'olive oil', 'onion'])
known_user = int(train['user_id'].iloc[0])

mode_cfg = QUERY_MODES['pantry']
recs = recommender.recommend(
    profile, context, user_id=known_user, top_n=5,
    weights_override=mode_cfg['weights'], mode='pantry',
)

print(f"=== {mode_cfg['label']} panel — {mode_cfg['description']} ===")
print(f"Pantry: {context.pantry}\n")
for i, rec in enumerate(recs, 1):
    mins = f"{rec.minutes} min" if rec.minutes else '?'
    print(f"{i}. {rec.name} [{mins}]")
    print(f"   pantry match: {rec.pantry_score:.0%}  |  {rec.explanation}")


In [ ]:
# 6b) Demo — Panel B: Time mode
# Signal: recipes within my time budget (hard cap applied)
# Validated by stage3_oracle_time: Hit@10 = 1.0
import colab_init
colab_init.bind("/content/cookmatch-recipe-recommender")

from config.query_modes import QUERY_MODES
from src.models.user_profile import UserProfile
from src.models.session_context import SessionContext

assert 'recommender' in globals(), 'Run cell 5 first'

MAX_MINUTES = 30
profile = UserProfile(diet='vegan', allergens=['nuts', 'dairy', 'gluten'])
context = SessionContext(max_minutes=MAX_MINUTES)
known_user = int(train['user_id'].iloc[0])

mode_cfg = QUERY_MODES['time']
recs = recommender.recommend(
    profile, context, user_id=known_user, top_n=5,
    weights_override=mode_cfg['weights'], mode='time',
)

print(f"=== {mode_cfg['label']} panel — {mode_cfg['description']} ===")
print(f"Max minutes: {MAX_MINUTES}\n")
for i, rec in enumerate(recs, 1):
    mins = f"{rec.minutes} min" if rec.minutes else '?'
    within = '✓' if (rec.minutes or 999) <= MAX_MINUTES else '✗'
    print(f"{i}. {rec.name} [{mins}] {within}")
    print(f"   {rec.explanation}")


In [ ]:
# 6c) Demo — Panel C: Intent mode
# Signal: recipes matching meal goal (main dish / snack / dessert / etc.)
# Validated by stage3_oracle_intent: Hit@10 = 1.0
import colab_init
colab_init.bind("/content/cookmatch-recipe-recommender")

from config.query_modes import QUERY_MODES
from src.models.user_profile import UserProfile
from src.models.session_context import SessionContext
from config.intents import MEAL_INTENT_TAGS

assert 'recommender' in globals(), 'Run cell 5 first'

INTENT = 'main'  # options: main, snack, dessert, breakfast, side, soup, salad
profile = UserProfile(diet='vegan', allergens=['nuts', 'dairy', 'gluten'])
context = SessionContext(meal_intent=INTENT)
known_user = int(train['user_id'].iloc[0])

mode_cfg = QUERY_MODES['intent']
recs = recommender.recommend(
    profile, context, user_id=known_user, top_n=5,
    weights_override=mode_cfg['weights'], mode='intent',
)

valid_intents = list(MEAL_INTENT_TAGS.keys())
print(f"=== {mode_cfg['label']} panel — {mode_cfg['description']} ===")
print(f"Intent: {INTENT}  (available: {valid_intents})\n")
for i, rec in enumerate(recs, 1):
    mins = f"{rec.minutes} min" if rec.minutes else '?'
    match = '✓' if rec.intent_score >= 1.0 else '~'
    print(f"{i}. {rec.name} [{mins}] {match}")
    print(f"   intent match: {rec.intent_score:.0%}  |  {rec.explanation}")


In [ ]:
# 6) Ablation eval (reuses cell 5 model)
import colab_init
colab_init.bind("/content/cookmatch-recipe-recommender")

import pandas as pd
import time

from src.eval.offline_eval import run_ablation

recipes_train = globals().get("recipes_train", recipes)
USER_SAMPLE = globals().get("USER_SAMPLE", 100)
TOP_K = 10

assert "recommender" in globals(), "Run cell 5 first"

print(f"Running ablation on {len(recipes_train)} recipes, {USER_SAMPLE} users...")
t0 = time.time()

results = run_ablation(
    recipes=recipes_train,
    train_interactions=train,
    held_out=validation,
    user_sample_size=USER_SAMPLE,
    k=TOP_K,
    stage3=recommender,
)

print(f"Ablation done in {(time.time()-t0)/60:.1f} min")

# --- Rating prediction baseline (separate from retrieval table) ---
rmse_row = results[results['scenario'] == 'mf_rating_prediction']
if not rmse_row.empty:
    rmse = rmse_row['rmse'].iloc[0]
    mae  = rmse_row['mae'].iloc[0]
    n    = int(rmse_row['n_predictions'].iloc[0])
    print(f"\nMF rating prediction baseline ({n} held-out interactions):")
    print(f"  RMSE : {rmse:.4f}")
    print(f"  MAE  : {mae:.4f}")
    print("  (for reference: comparable system RMSE=1.777, MAE=1.087)\n")

# --- Retrieval + diversity table (hide mf_top500 and rating prediction row) ---
HIDE = {'stage2_mf_top500', 'mf_rating_prediction'}
display_results = results[~results['scenario'].isin(HIDE)]
display(display_results.sort_values(by=f"hit_rate@{TOP_K}", ascending=False))
results.to_csv("ablation_results.csv", index=False)
print("Saved ablation_results.csv")

## Full catalog run

Uncomment below only if you accept long runtimes:

```python
recipes_full = load_recipes(columns=["id", "name", "ingredients", "minutes", "tags"])
recommender = Stage3Recommender().fit(recipes_full, train)
results_full = run_ablation(recipes=recipes_full, train_interactions=train, held_out=validation, user_sample_size=500, k=10)
results_full.to_csv("ablation_full.csv", index=False)
```